# Evaluasi Arsitektur NLP: Simple RNN vs LSTM vs Transformer (From Scratch)
### Studi Kasus: Klasifikasi Berita BBC News berdasarkan Variasi Panjang Teks (Sequence Length)

Notebook ini berisi seluruh kode pengujian dan perbandingan performa tiga arsitektur deep learning utama untuk NLP:
1. **Simple RNN (Elman RNN)**
2. **LSTM (Long Short-Term Memory)**
3. **Transformer Classifier (Encoder-only)**

Ketiga model di atas dibangun **dari awal (from scratch)** menggunakan PyTorch tanpa menggunakan library tingkat tinggi seperti Hugging Face Transformers untuk modelnya. Dataset yang digunakan adalah **BBC News** dari Hugging Face (`SetFit/bbc-news`) dengan 5 kategori berita. Pengujian dilakukan pada 4 skenario panjang teks (sequence length): **50, 200, 500, dan 1000 token** untuk melihat pengaruh panjang teks terhadap performa masing-masing arsitektur.


## 1. Pustaka & Verifikasi Perangkat Keras
Menginstal pustaka yang diperlukan (untuk Google Colab) dan mengonfigurasi GPU akselerasi jika tersedia.


In [ ]:
# Instalasi library jika belum terpasang (misal di Google Colab)
try:
    import datasets
    import sklearn
    import seaborn
except ImportError:
    print("Menginstal pustaka yang diperlukan...")
    !pip install -q torch datasets scikit-learn matplotlib seaborn pandas tqdm

import os
import math
import time
import json
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

print("Pustaka berhasil diimpor!")


## 2. Konfigurasi Eksperimen
Mendefinisikan parameter konfigurasi pusat untuk dataset, hyperparameter model, skenario sequence length, dan reproduksibilitas.


In [ ]:
# Perangkat komputasi (GPU CUDA jika tersedia, else CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device komputasi yang digunakan: {DEVICE}")

# Seed acak untuk hasil yang konsisten
RANDOM_SEED = 42

# Konfigurasi Dataset
DATASET_NAME = "SetFit/bbc-news"
NUM_CLASSES = 5
LABEL_MAP = {
    0: "business",
    1: "entertainment",
    2: "politics",
    3: "sport",
    4: "tech"
}

# Parameter Vocabulary & Preprocessing
VOCAB_SIZE = 20000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

# Hyperparameter Arsitektur Model (Sama untuk perbandingan yang adil)
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

# Parameter Khusus Transformer
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF_DIM = 256
TRANSFORMER_DROPOUT = 0.1

# Hyperparameter Training
BATCH_SIZE = 64
NUM_EPOCHS = 8
LEARNING_RATE = 1e-3
PATIENCE = 2

# Skenario Panjang Sequence yang Diuji
SEQ_LENGTHS = [50, 200, 500, 1000]

# Setup folder penyimpanan hasil
RESULTS_DIR = "./results"
PLOTS_DIR = "./plots"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)


## 3. Preprocessing Data & Tokenisasi
Membangun kelas `Vocabulary` kustom, kelas `BBCNewsDataset` PyTorch, dan fungsi `get_data_loaders` untuk memuat dataset berita BBC dan membaginya menjadi set training, validation, dan test secara stratifikasi.


In [ ]:
class Vocabulary:
    def __init__(self, vocab_size=VOCAB_SIZE):
        self.vocab_size = vocab_size
        self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
        self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN}
        self.word_counts = {}

    def tokenize(self, text):
        text = text.lower()
        cleaned = ""
        for char in text:
            if char.isalnum() or char.isspace():
                cleaned += char
            else:
                cleaned += " "
        tokens = [t for t in cleaned.split() if t]
        return tokens

    def build_vocab(self, texts):
        for text in texts:
            tokens = self.tokenize(text)
            for token in tokens:
                self.word_counts[token] = self.word_counts.get(token, 0) + 1

        sorted_words = sorted(self.word_counts.items(), key=lambda x: x[1], reverse=True)
        for word, _ in sorted_words[:self.vocab_size - 2]:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, tokens, max_len):
        truncated_tokens = tokens[:max_len]
        indices = [self.word2idx.get(t, self.word2idx[UNK_TOKEN]) for t in truncated_tokens]
        padding_len = max_len - len(indices)
        indices += [self.word2idx[PAD_TOKEN]] * padding_len
        return indices


class BBCNewsDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        tokens = self.vocab.tokenize(text)
        encoded = self.vocab.encode(tokens, self.max_len)
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )


def get_data_loaders(max_seq_len):
    print(f"🔄 Memuat dataset BBC News dari Hugging Face...")
    ds = load_dataset(DATASET_NAME)
    
    train_data = ds["train"]
    test_data = ds["test"]
    
    train_texts = train_data["text"]
    train_labels = train_data["label"]
    test_texts = test_data["text"]
    test_labels = test_data["label"]
    
    # Validation split 15% dari train set secara stratified
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts,
        train_labels,
        test_size=0.15,
        random_state=RANDOM_SEED,
        stratify=train_labels
    )
    
    # Bangun vocabulary dari training data saja
    vocab = Vocabulary(vocab_size=VOCAB_SIZE)
    vocab.build_vocab(train_texts)
    print(f"📊 Jumlah kata unik terdaftar di Vocab: {len(vocab.word2idx)} / {VOCAB_SIZE}")
    
    train_dataset = BBCNewsDataset(train_texts, train_labels, vocab, max_seq_len)
    val_dataset = BBCNewsDataset(val_texts, val_labels, vocab, max_seq_len)
    test_dataset = BBCNewsDataset(test_texts, test_labels, vocab, max_seq_len)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    
    print(f"[OK] Data loaders berhasil dibuat (Sequence Length Max: {max_seq_len})")
    print(f"   - Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}")
    
    return train_loader, val_loader, test_loader, len(vocab.word2idx)


## 4. Model Arsitektur 1 — Simple RNN (Elman RNN)
Mengimplementasikan klasifikasi berbasis Recurrent Neural Network sederhana. Mengatasi bug padding dengan mengekstraksi hidden state pada indeks timestep kata asli terakhir.


In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx=0):
        super(SimpleRNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        out, h_n = self.rnn(embedded)
        
        # Cari token non-padding terakhir
        mask = (x != self.embedding.padding_idx)
        lengths = mask.sum(dim=1).clamp(min=1)
        batch_indices = torch.arange(out.size(0), device=out.device)
        last_hidden = out[batch_indices, lengths - 1, :]
        
        logits = self.fc(last_hidden)
        return logits


## 5. Model Arsitektur 2 — LSTM (Long Short-Term Memory)
Mengimplementasikan LSTM Classifier yang memiliki cell state dan gating mechanisms untuk mengatasi masalah vanishing gradient.


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx=0):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        out, (h_n, c_n) = self.lstm(embedded)
        
        # Cari token non-padding terakhir
        mask = (x != self.embedding.padding_idx)
        lengths = mask.sum(dim=1).clamp(min=1)
        batch_indices = torch.arange(out.size(0), device=out.device)
        last_hidden = out[batch_indices, lengths - 1, :]
        
        logits = self.fc(last_hidden)
        return logits


## 6. Model Arsitektur 3 — Transformer Encoder Classifier
Mengimplementasikan Transformer Encoder secara mandiri (*from scratch*) yang terdiri dari:
1. **Positional Encoding** (sinusoidal).
2. **Multi-Head Self-Attention**.
3. **Position-wise Feed-Forward Network**.
4. **Transformer Encoder Block** dengan residual connection dan layer normalization.
5. **Mask-Aware Global Average Pooling** untuk klasifikasi akhir yang terhindar dari bias padding.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()
        
        q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.out_linear(context)
        return output


class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()
        self.attention = MultiHeadSelfAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        attn_out = self.attention(x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_out))
        return x


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, d_ff, num_layers, num_classes, max_len=1000, pad_idx=0, dropout=0.1):
        super(TransformerClassifier, self).__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(embedding_dim, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(embedding_dim, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(embedding_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != self.pad_idx).unsqueeze(1).unsqueeze(2).to(x.device)
        embedded = self.embedding(x)
        x = self.pos_encoder(embedded)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask)
            
        # Mask-Aware Global Average Pooling
        mask_squeezed = mask.squeeze(1).squeeze(1).float()
        summed = (x * mask_squeezed.unsqueeze(-1)).sum(dim=1)
        lengths = mask_squeezed.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = summed / lengths
        
        logits = self.fc(pooled)
        return logits


## 7. Fungsi Pembantu Training & Evaluasi
Mendefinisikan fungsi `train_model` yang dilengkapi dengan tracking loss/akurasi, perhitungan durasi latihan, serta **Early Stopping** untuk mencegah overfitting (menyimpan bobot validasi terbaik). Dan fungsi `evaluate_model` untuk mengukur performa pengujian secara makro.


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels).item()
        total_samples += labels.size(0)
        
    return total_loss / total_samples, correct_predictions / total_samples


def evaluate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            logits = model(inputs)
            loss = criterion(logits, labels)
            
            total_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels).item()
            total_samples += labels.size(0)
            
    return total_loss / total_samples, correct_predictions / total_samples


def train_model(model, train_loader, val_loader, epochs, lr, patience, device):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    best_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    
    epochs_no_improve = 0
    start_time = time.time()
    
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate_epoch(model, val_loader, criterion, device)
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        # Cek model terbaik
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"  [Early Stopping] Epoch {epoch:02d} dipicu.")
            break
            
    total_duration = time.time() - start_time
    model.load_state_dict(best_model_wts)
    return model, total_duration, history


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_model(model, test_loader, device):
    model.eval()
    all_predictions = []
    all_true_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            _, preds = torch.max(logits, dim=1)
            all_predictions.extend(preds.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_true_labels, all_predictions)
    precision = precision_score(all_true_labels, all_predictions, average='macro', zero_division=0)
    recall = recall_score(all_true_labels, all_predictions, average='macro', zero_division=0)
    f1 = f1_score(all_true_labels, all_predictions, average='macro', zero_division=0)
    
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }


## 8. Eksekusi Eksperimen Perbandingan Model
Menjalankan simulasi training untuk Simple RNN, LSTM, dan Transformer Classifier pada variasi sequence length (50, 200, 500, 1000) dan mengumpulkan hasilnya.


In [ ]:
# Fungsi menjamin reproducibility
def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
all_results = []

# Loop skenario sequence length
for max_len in SEQ_LENGTHS:
    print("\n" + "=" * 65)
    print(f"🌀 SKENARIO PANJANG MAKSIMUM TEKS: {max_len} TOKEN")
    print("=" * 65)
    
    train_loader, val_loader, test_loader, vocab_size = get_data_loaders(max_len)
    
    models_to_test = {
        "SimpleRNN": lambda: SimpleRNNClassifier(
            vocab_size=vocab_size,
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            num_classes=NUM_CLASSES,
            pad_idx=0
        ),
        "LSTM": lambda: LSTMClassifier(
            vocab_size=vocab_size,
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            num_classes=NUM_CLASSES,
            pad_idx=0
        ),
        "Transformer": lambda: TransformerClassifier(
            vocab_size=vocab_size,
            embedding_dim=EMBEDDING_DIM,
            num_heads=TRANSFORMER_HEADS,
            d_ff=TRANSFORMER_FF_DIM,
            num_layers=TRANSFORMER_LAYERS,
            num_classes=NUM_CLASSES,
            max_len=max_len,
            pad_idx=0,
            dropout=TRANSFORMER_DROPOUT
        )
    }
    
    for model_name, model_fn in models_to_test.items():
        print(f"📦 Melatih Model: {model_name} (Max Seq Len: {max_len})")
        model = model_fn()
        num_params = count_parameters(model)
        
        trained_model, training_time, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            patience=PATIENCE,
            device=DEVICE
        )
        
        metrics = evaluate_model(trained_model, test_loader, DEVICE)
        print(f"   Accuracy: {metrics['accuracy']*100:.2f}% | F1-Score: {metrics['f1_score']*100:.2f}% | Waktu: {training_time:.2f}s")
        
        result_entry = {
            "model_name": model_name,
            "seq_length": max_len,
            "parameters": num_params,
            "training_time_seconds": round(training_time, 2),
            "accuracy": round(metrics["accuracy"], 4),
            "precision": round(metrics["precision"], 4),
            "recall": round(metrics["recall"], 4),
            "f1_score": round(metrics["f1_score"], 4)
        }
        all_results.append(result_entry)
        
        del model, trained_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Simpan hasil dalam JSON
with open(os.path.join(RESULTS_DIR, "experiment_results.json"), "w") as f:
    json.dump(all_results, f, indent=4)
    
df = pd.DataFrame(all_results)
df_display = df.copy()
df_display["accuracy"] = (df_display["accuracy"] * 100).apply(lambda x: f"{x:.2f}%")
df_display["precision"] = (df_display["precision"] * 100).apply(lambda x: f"{x:.2f}%")
df_display["recall"] = (df_display["recall"] * 100).apply(lambda x: f"{x:.2f}%")
df_display["f1_score"] = (df_display["f1_score"] * 100).apply(lambda x: f"{x:.2f}%")
df_display["training_time_seconds"] = df_display["training_time_seconds"].apply(lambda x: f"{x:.2f}s")
df_display["parameters"] = df_display["parameters"].apply(lambda x: f"{x:,}")

print("\n" + "=" * 90)
print("📊 TABEL PERBANDINGAN AKHIR EKSPERIMEN (RNN vs LSTM vs TRANSFORMER)")
print("=" * 90)
print(df_display.to_string(index=False))
print("=" * 90)


## 9. Visualisasi Hasil Eksperimen
Menyajikan grafik perbandingan performa akurasi, F1-score, dan total waktu pelatihan antar-model berdasarkan variasi sequence length secara interaktif.


In [ ]:
# Load data hasil jika diperlukan kembali
with open(os.path.join(RESULTS_DIR, "experiment_results.json"), "r") as f:
    results_data = json.load(f)
df = pd.DataFrame(results_data)

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 16,
    "legend.fontsize": 10,
    "figure.dpi": 150
})

palette_colors = {
    "SimpleRNN": "#E05A47",
    "LSTM": "#FFA600",
    "Transformer": "#58508D"
}

# 1. Perbandingan Akurasi (Bar Chart)
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df, x="seq_length", y="accuracy", hue="model_name", palette=palette_colors, edgecolor="black", linewidth=0.8)
plt.title("Perbandingan Accuracy Berdasarkan Panjang Teks (Sequence Length)", pad=15, fontweight="bold")
plt.xlabel("Panjang Maksimum Token (Sequence Length)", labelpad=10)
plt.ylabel("Accuracy", labelpad=10)
plt.ylim(0, 1.05)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height*100:.1f}%", (p.get_x() + p.get_width() / 2., height + 0.015), ha='center', va='center', xytext=(0, 3), textcoords='offset points', fontsize=8, fontweight="bold")
plt.legend(title="Arsitektur Model", loc="lower right")
plt.tight_layout()
plt.show()

# 2. Perbandingan F1-Score (Bar Chart)
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df, x="seq_length", y="f1_score", hue="model_name", palette=palette_colors, edgecolor="black", linewidth=0.8)
plt.title("Perbandingan Macro F1-Score Berdasarkan Panjang Teks", pad=15, fontweight="bold")
plt.xlabel("Panjang Maksimum Token (Sequence Length)", labelpad=10)
plt.ylabel("Macro F1-Score", labelpad=10)
plt.ylim(0, 1.05)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height*100:.1f}%", (p.get_x() + p.get_width() / 2., height + 0.015), ha='center', va='center', xytext=(0, 3), textcoords='offset points', fontsize=8, fontweight="bold")
plt.legend(title="Arsitektur Model", loc="lower right")
plt.tight_layout()
plt.show()

# 3. Perbandingan Waktu Pelatihan (Bar Chart)
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df, x="seq_length", y="training_time_seconds", hue="model_name", palette=palette_colors, edgecolor="black", linewidth=0.8)
plt.title("Perbandingan Total Waktu Pelatihan (Training Time)", pad=15, fontweight="bold")
plt.xlabel("Panjang Maksimum Token (Sequence Length)", labelpad=10)
plt.ylabel("Waktu Pelatihan (Detik)", labelpad=10)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.1f}s", (p.get_x() + p.get_width() / 2., height + max(df['training_time_seconds'])*0.015), ha='center', va='center', xytext=(0, 3), textcoords='offset points', fontsize=8, fontweight="bold")
plt.legend(title="Arsitektur Model", loc="upper left")
plt.tight_layout()
plt.show()

# 4. Tren Akurasi vs Sequence Length (Line Chart)
plt.figure(figsize=(10, 5))
markers = {"SimpleRNN": "o", "LSTM": "s", "Transformer": "^"}
for model in df["model_name"].unique():
    model_df = df[df["model_name"] == model]
    plt.plot(model_df["seq_length"], model_df["accuracy"], marker=markers.get(model, "o"), markersize=8, linewidth=2.5, color=palette_colors[model], label=model)
plt.title("Tren Perubahan Accuracy Terhadap Panjang Teks", pad=15, fontweight="bold")
plt.xlabel("Panjang Maksimum Token (Sequence Length)", labelpad=10)
plt.ylabel("Accuracy", labelpad=10)
plt.xticks(df["seq_length"].unique())
plt.ylim(min(df["accuracy"]) - 0.05, 1.02)
plt.legend(title="Arsitektur Model", loc="lower right")
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 10. Analisis Akademis (Bahan Tanya Jawab Ujian Lisan PBA)

Di bawah ini adalah analisis teoretis mendalam mengenai performa ketiga arsitektur berdasarkan hasil eksperimen untuk menjawab pertanyaan kritis dalam ujian lisan.

### ❓ Pertanyaan 1: Mengapa Simple RNN mengalami penurunan performa ketika panjang teks bertambah?
* **Jawaban:** Simple RNN memproses token secara sekuensial. Selama proses pembelajaran balik (*backpropagation through time* / BPTT), gradien error ditransmisikan ke masa lalu melalui perkalian berantai matriks bobot berulang $W_{hh}$. Secara matematis, jika nilai eigen terbesar dari $W_{hh}$ kurang dari 1, perkalian berantai ini menyebabkan gradien menyusut secara eksponensial mendekati nol (*vanishing gradient*). Akibatnya, bobot untuk timestep awal tidak diperbarui secara efektif, menyebabkan model "lupa" konteks penting di awal teks. Selain itu, representasi semua informasi dikompresi ke dalam satu hidden state berdimensi tetap (*information bottleneck*), yang menyulitkan penyimpanan informasi detail dari sequence yang sangat panjang (seperti 1000 token).

### ❓ Pertanyaan 2: Bagaimana LSTM mengatasi sebagian masalah RNN?
* **Jawaban:** LSTM memperkenalkan **Cell State ($c_t$)** dan tiga mekanisme gerbang (**Forget, Input, dan Output gates**). Cell state bertindak sebagai jalur linier yang mengalirkan informasi dengan gangguan non-linear minimal, membentuk *Constant Error Carousel* (CEC). Hal ini memungkinkan gradien mengalir kembali dengan stabil tanpa penyusutan eksponensial. Forget gate mengontrol informasi apa yang dibuang, input gate mengontrol informasi baru apa yang disimpan, dan output gate mengontrol bagian cell state mana yang diekspos sebagai hidden state. Walaupun LSTM jauh lebih tangguh menghadapi teks panjang dibanding RNN, LSTM tetap sekuensial; sehingga untuk sequence yang sangat panjang (1000+ token), ia masih dapat mengalami penurunan performa bertahap akibat batas kapasitas representation compression.

### ❓ Pertanyaan 3: Mengapa Transformer tetap stabil pada teks panjang?
* **Jawaban:** Transformer stabil karena menggantikan model sekuensial dengan komputasi paralel langsung:
  1. **Jalur Komputasi O(1):** Jarak maksimum antara token apa pun di dalam sequence untuk saling berinteraksi adalah konstan (satu langkah), sehingga tidak ada distorsi informasi akibat langkah sekuensial.
  2. **Stabilitas Gradien:** Tanpa adanya rekurensi (BPTT), gradien dapat langsung mengalir secara instan dari output ke representasi kata mana pun melalui koneksi atensi (*self-attention*) dan koneksi residu (*residual connection*).
  3. **Positional Encoding:** Memberikan representasi matematis posisi absolut yang stabil dan independen terhadap panjang teks.

### ❓ Pertanyaan 4: Bagaimana self-attention membantu Transformer memahami konteks global?
* **Jawaban:** Melalui mekanisme **Scaled Dot-Product Attention**:
  $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
  1. Proyeksi $Q$ (Query), $K$ (Key), dan $V$ (Value) dibentuk secara linier untuk setiap kata.
  2. Perkalian matriks korelasi $Q K^T$ membandingkan relevansi setiap kata dengan seluruh kata lain di seluruh teks.
  3. Pembagi $\sqrt{d_k}$ menstabilkan distribusi softmax agar gradien tidak jenuh.
  4. Fungsi softmax mengubah skor korelasi menjadi bobot atensi (probabilitas relevansi).
  5. Hasil perkalian dengan $V$ memadukan konteks global kata-kata terkait ke dalam representasi token saat ini secara dinamis.
  Multi-Head Attention memungkinkan model secara simultan berfokus pada berbagai relasi bahasa yang berbeda (misalnya hubungan sintaksis vs semantis jarak jauh).

### ❓ Pertanyaan 5: Apakah peningkatan performa Transformer sebanding dengan kompleksitas dan waktu pelatihannya?
* **Jawaban:** **YA, sangat sebanding**.
  - **Efisiensi GPU:** Walaupun kompleksitas komputasi self-attention secara teoritis adalah kuadratik terhadap panjang teks ($O(N^2)$), semua token diproses secara paralel, bukan sekuensial. Hal ini memungkinkan pemanfaatan optimal penuh akselerasi GPU.
  - **Skalabilitas & Ketangguhan:** Pada teks panjang (500 - 1000 token), performa akurasi Simple RNN dan LSTM anjlok secara drastif, sedangkan Transformer tetap mempertahankan kestabilan performa akurasi yang tinggi. Investasi tambahan waktu komputasi yang relatif kecil menghasilkan peningkatan akurasi dan keandalan model yang sangat signifikan.
